# 04. Full lesson: lokalny LLM, RAG, ChromaDB, workflow i trening w jednym notebooku

To jest wersja demonstracyjna „wszystko w jednym” na zajęcia.

**Kolejność:**

1. instalacja,
2. lokalny LLM w Colabie,
3. mini RAG TF-IDF,
4. RAG z ChromaDB,
5. router workflow,
6. mała sieć neuronowa do klasyfikacji ścieżki.

In [ ]:
!pip -q install transformers accelerate sentencepiece sentence-transformers chromadb scikit-learn pandas

In [ ]:
import os
import shutil
import torch
import pandas as pd
import numpy as np

print("GPU dostępne:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Część A. Lokalny LLM w Colabie

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Urządzenie:", device)

# Uwaga: na GPU używamy float16, na CPU float32.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
model.to(device)
model.eval()

print("Model załadowany:", MODEL_NAME)

In [ ]:
def ask_llm(prompt, system="Jesteś pomocnym asystentem dydaktycznym. Odpowiadasz po polsku, krótko i precyzyjnie.", max_new_tokens=250):
    """Prosta funkcja do rozmowy z lokalnym modelem uruchomionym w Colabie."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    return answer.strip()

In [ ]:
print(ask_llm("W 5 punktach wyjaśnij, czym jest lokalny LLM."))

# Część B. Dokumenty dla RAG

In [ ]:
documents = [
    "Student może mieć maksymalnie dwie nieobecności.",
    "Trzecia nieobecność wymaga wykonania zadania dodatkowego.",
    "Projekt końcowy musi zostać oddany do 30 czerwca.",
    "Projekt może być wykonany indywidualnie albo w parach.",
    "Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.",
    "Plagiat powoduje brak zaliczenia.",
    "Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia."
]

for i, d in enumerate(documents):
    print(i, d)

# Część C. Mini RAG TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_tfidf(query, documents, n_results=2):
    vectorizer = TfidfVectorizer()
    doc_vectors = vectorizer.fit_transform(documents)
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, doc_vectors).flatten()
    best_indices = similarities.argsort()[::-1][:n_results]
    return [(documents[i], float(similarities[i])) for i in best_indices]

def rag_tfidf(query, documents, n_results=2):
    selected = retrieve_tfidf(query, documents, n_results=n_results)
    selected_docs = [doc for doc, score in selected]
    context = " ".join(selected_docs)

    prompt = f"""Odpowiedz na pytanie wyłącznie na podstawie kontekstu.

KONTEKST:
{context}

PYTANIE:
{query}

Jeżeli w kontekście nie ma odpowiedzi, napisz:
"Nie wiem na podstawie dostarczonych dokumentów."
"""
    answer = ask_llm(prompt, max_new_tokens=180)
    return answer, selected

query = "Czy mogę użyć ChatGPT w projekcie?"
answer, selected = rag_tfidf(query, documents)

print("FRAGMENTY:")
for doc, score in selected:
    print(f"{score:.3f} | {doc}")

print("ODPOWIEDŹ:")
print(answer)

# Część D. RAG z ChromaDB

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = embedding_model.encode(documents).tolist()

DB_PATH = "rag_db_full"
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)

chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_or_create_collection("regulamin")

collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings
)

print("ChromaDB gotowa.")

In [ ]:
def retrieve_chroma(query, collection, embedding_model, n_results=3):
    query_embedding = embedding_model.encode([query]).tolist()[0]
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)
    return results["documents"][0]

def rag_chroma(query, collection, embedding_model, n_results=3):
    retrieved_docs = retrieve_chroma(query, collection, embedding_model, n_results=n_results)
    context = " ".join(retrieved_docs)

    prompt = f"""
Odpowiedz na pytanie wyłącznie na podstawie kontekstu.

KONTEKST:
{context}

PYTANIE:
{query}

ZASADY:
- Nie dopowiadaj niczego spoza kontekstu.
- Jeżeli odpowiedzi nie ma w kontekście, napisz: "Nie wiem na podstawie dostarczonych dokumentów."
- Odpowiedz po polsku.
"""
    answer = ask_llm(prompt, max_new_tokens=200)
    return answer, retrieved_docs

for q in [
    "Czy student z trzema nieobecnościami może zaliczyć kurs?",
    "Czy projekt można robić w parach?",
    "Czy egzamin jest ustny?"
]:
    print("=" * 90)
    print("PYTANIE:", q)
    answer, docs = rag_chroma(q, collection, embedding_model)
    print("FRAGMENTY:")
    for d in docs:
        print("-", d)
    print("ODPOWIEDŹ:")
    print(answer)

# Część E. Router workflow

In [ ]:
def workflow_router(question):
    q = question.lower()
    document_words = ["dokument", "pdf", "regulamin", "instrukcja", "procedura", "sylabus", "umowa"]
    calculation_words = ["policz", "oblicz", "średnia", "suma", "wykres", "csv", "excel", "tabela"]
    training_words = ["trening", "trenować", "dostroić", "fine-tuning", "nauczyć model", "mój styl"]

    if any(word in q for word in document_words):
        return "RAG", "Pytanie dotyczy dokumentów lub bazy wiedzy."
    elif any(word in q for word in calculation_words):
        return "TOOL", "Pytanie wymaga obliczeń albo pracy na danych."
    elif any(word in q for word in training_words):
        return "TRAINING", "Pytanie dotyczy uczenia lub dostrajania modelu."
    else:
        return "PROMPT", "Wystarczy zwykła odpowiedź modelu."

for case in [
    "Wyjaśnij mi, czym jest LLM.",
    "Mam dokument PDF i chcę pytać o jego treść.",
    "Policz średnią ocen z pliku CSV.",
    "Chcę dostroić model do mojego stylu pisania."
]:
    category, reason = workflow_router(case)
    print(case, "=>", category, "|", reason)

# Część F. Mała sieć neuronowa do klasyfikacji workflow

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

texts = [
    "Wyjaśnij czym jest sztuczna inteligencja", "Napisz krótką definicję LLM",
    "Co to jest transformer", "Wyjaśnij embedding prostym językiem",
    "Mam dokument PDF i chcę zadawać pytania", "Odpowiedz na podstawie regulaminu",
    "Mam bazę wiedzy i chcę z niej korzystać", "Chcę pytać o treść instrukcji BHP",
    "Policz średnią ocen studentów", "Wykonaj analizę danych z pliku CSV",
    "Oblicz odchylenie standardowe", "Zrób wykres sprzedaży miesięcznej",
    "Chcę, żeby model odpowiadał zawsze w stylu prawniczym",
    "Chcę dostroić model do mojego tonu wypowiedzi",
    "Chcę nauczyć model klasyfikować moje teksty", "Potrzebuję trenować model na przykładach"
]
labels = [
    "PROMPT", "PROMPT", "PROMPT", "PROMPT",
    "RAG", "RAG", "RAG", "RAG",
    "TOOL", "TOOL", "TOOL", "TOOL",
    "TRAINING", "TRAINING", "TRAINING", "TRAINING"
]

classifier = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("mlp", MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42))
])
classifier.fit(texts, labels)

questions = [
    "Mam dokumenty uczelni i chcę zadawać pytania o zasady zaliczenia",
    "Policz sumę sprzedaży z tabeli",
    "Wyjaśnij, czym jest RAG",
    "Chcę nauczyć model rozpoznawania mojego stylu"
]

for question in questions:
    print(question, "=>", classifier.predict([question])[0])

# Podsumowanie

Największy błąd początkujących: próbować wszystko trenować. W większości przypadków zaczynamy od workflow: prompt, RAG, narzędzia, walidacja, a dopiero potem ewentualny fine-tuning.